# FLAG 2027 - Notebook 39 RUN-ALL: new physical cues for the same-gender cells (research round 8, part 2)

Everything that changes HOW the head is trained has now been tried (NB27, 28, 31, 33, 34, 36, 36b, 37, 38) and only two kinds of change ever reached the board: a loss form (InfoNCE, L_out) and new information (Whisper block, NAP). This notebook adds information of a kind we do not have yet. Within one gender, the one face-voice link with a physical mechanism is skull and vocal-tract geometry: nose, mouth and face width against the resonances of the voice (Speech2Face, Cross-Modal Perceptionist). Our face input is one CLIP vector of the whole image; our voice inputs are speaker-verification embeddings plus the deepest Whisper layers, all trained to discard the low-level acoustics that carry vocal-tract shape.

| Name | Block | What it adds |
|---|---|---|
| `F_geo` | 68 3D landmarks (InsightFace buffalo_l, frozen) -> Procrustes-aligned shape (204-d) + 15 craniofacial ratios (face width and height, nose length and width, mouth width, lip thickness, jaw width, nose depth, ...) | explicit face geometry, no identity texture |
| `F_nm` | CLIP ViT-L/14 on the nose-mouth-chin crop of the detected face | CLIP looking at the lower face only |
| `V_whm` | Whisper-large-v3 encoder layers 8-16 (we already use 20-28) | mid-level acoustics (formants, timbre) before the speaker and language abstraction |
| `V_wlo` | WavLM-large layers 3-8, mean over time | low-level self-supervised acoustics (pitch, vocal-tract length cues) |
| `V_wmid` | WavLM-large layers 12-18 | mid-level self-supervised speech representation |

Each block is added to system E's inputs with E's loss; reference `R0_ref` is copied from NB36. Stage A = usual folds, seeds 1-3; top 3 re-checked on NEW folds with NEW seeds (stage B, the only one that counts); stage C = all adopted blocks together. Extraction covers the challenge train rows and the MAV rows only (what the CV uses). A winning block would then need the same extraction on the two VoxCeleb parts system E uses before it can go into NB20.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 3.5-4 h (faces ~25 min, voices ~1.5-2 h, CV ~1.2 h). Resumable: every block and split is saved as it finishes. Paste the **SUMMARY**.


In [ ]:
# Cell 1 [NB39 v1 5-Nov] - Drive, GPU (hard check), InsightFace (frozen landmark model, declared in the system description)
import os, importlib.util
if importlib.util.find_spec('insightface') is None:
    os.system('pip -q install insightface onnxruntime-gpu')        # builds a small C extension, ~3-5 min the first time
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil, onnxruntime
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
print('[NB39 v1 5-Nov] Device:', torch.cuda.get_device_name(0), '| onnxruntime providers:', onnxruntime.get_available_providers(), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [NB39 v1 5-Nov] - face AND voice file lists in our row order (challenge train, MAV v1/v3, MAV extra; dev rows are not needed for CV)
#          + alignment checks: CLIP on 100 faces per split and Whisper layers 20-28 on 48 voices per split must reproduce the stored features
import os, re, json, time, zipfile
import numpy as np, pandas as pd, cv2, librosa
from PIL import Image
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
PARTS = os.path.join(FEATDIR, 'nb39_parts'); os.makedirs(PARTS, exist_ok=True)
EXTRACT_NOTES = []
def enote(msg):
    print(msg); EXTRACT_NOTES.append(msg)
# ---- challenge train: faces and voices ----
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
if not os.path.exists(os.path.join(FULL, 'train_set.zip.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(('.jpg', '.jpeg', '.png', '.wav', '.txt', '.csv'))])
    open(os.path.join(FULL, 'train_set.zip.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def tok(tokens, exts, fallback=3):
    t = [x for x in tokens if x.lower().endswith(exts)]; return t[0] if len(t) == 1 else tokens[fallback]   # NB35 used token 3 for the face; NB24 the single .wav token
IMG, WAV = ('.jpg', '.jpeg', '.png'), ('.wav',)
SPLITS = ['train', 'mav', 'extra']
lp = find_one('train_English.txt'); lines = [l.split() for l in open(lp) if l.strip()]
FACES = {'train': [resolve(tok(t, IMG), lp) for t in lines]}; WAVS = {'train': [resolve(tok(t, WAV), lp) for t in lines]}
# ---- MAV v1/v3 (same rows as mavceleb_v1v3.npz) ----
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_f, mav_v, mspk = [], [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.both.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver)); zf.extract(fm, os.path.join(MAVD, ver))
            mav_v.append(os.path.join(MAVD, ver, vm)); mav_f.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FACES['mav'], WAVS['mav'] = mav_f, mav_v
# ---- MAV v2 + v1-complete 'extra' rows: the SAME random face per clip as the original extraction (same rng sequence as NB24/NB29/NB35) ----
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'both.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(vn, D); zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn), os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
WAVS['extra'] = [r[1] for r, k in zip(rows, keep) if k]; FACES['extra'] = [r[2] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
for s in SPLITS:
    assert None not in FACES[s] and None not in WAVS[s], 'missing face/voice files in %s - tell Claude' % s
    assert len(FACES[s]) == len(WAVS[s])
enote('rows: ' + ', '.join('%s %d' % (s, len(FACES[s])) for s in SPLITS))
def cos_rows(a, b):
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)
# ---- alignment check 1: CLIP on 100 random faces per split must reproduce the stored CLIP rows ----
from transformers import CLIPModel, CLIPProcessor
CLIPM = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').half().to(dev).eval(); CLIPP = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
@torch.no_grad()
def clip_embed(pils):
    x = CLIPP(images=pils, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
    return CLIPM.visual_projection(CLIPM.vision_model(pixel_values=x).pooler_output).float().cpu().numpy()
CLIP_SAVED = np.load(os.path.join(FEATDIR, 'clip.npz'))
stored_f = {'train': CLIP_SAVED['train'], 'mav': MAV['clip'], 'extra': EXTRA['clip']}
rng_c = np.random.default_rng(1)
for s in SPLITS:
    saved = stored_f[s]; assert len(saved) == len(FACES[s]), '%s: %d stored CLIP rows vs %d face files - tell Claude' % (s, len(saved), len(FACES[s]))
    idx = rng_c.choice(len(saved), 100, replace=False)
    med = np.median(cos_rows(clip_embed([Image.open(FACES[s][i]).convert('RGB') for i in idx]), saved[idx]))
    enote('alignment faces  %-6s median CLIP cosine %.4f' % (s, med)); assert med > 0.98, 'face files do not match the stored CLIP rows for %s - tell Claude' % s
# ---- alignment check 2: Whisper layers 20-28 on 48 random voices per split must reproduce the stored whisper rows (same code as NB24) ----
def get_whisper():
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    return WhisperFeatureExtractor.from_pretrained(name), WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval()
def whisper_batch(fe, enc, paths, bands):
    """one batch of clips -> one pooled vector per band of layers (mean of layers lo..hi-1, mean over the clip's real frames)"""
    wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths]
    x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
    with torch.no_grad():
        hs = enc(x, output_hidden_states=True).hidden_states
    outs = []
    for lo, hi in bands:
        h = torch.stack(hs[lo:hi]).float().mean(0); o = []
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); o.append(h[k_, :n].mean(0).cpu().numpy())
        outs.append(np.stack(o))
    return outs
WFE, WENC = get_whisper()
WH_SAVED = np.load(os.path.join(FEATDIR, 'whisper.npz'))
rng_w = np.random.default_rng(2)
for s in SPLITS:
    saved = WH_SAVED[s]; assert len(saved) == len(WAVS[s]), '%s: %d stored whisper rows vs %d voice files - tell Claude' % (s, len(saved), len(WAVS[s]))
    idx = rng_w.choice(len(saved), 48, replace=False)
    new = np.concatenate([whisper_batch(WFE, WENC, [WAVS[s][i] for i in idx[b:b + 8]], [(20, 29)])[0] for b in range(0, 48, 8)])
    med = np.median(cos_rows(new, saved[idx])); enote('alignment voices %-6s median Whisper cosine %.4f' % (s, med)); assert med > 0.98, 'voice files do not match the stored whisper rows for %s - tell Claude' % s


In [ ]:
# Cell 3 [NB39 v1 5-Nov] - FACE cues. 'geo': 68 3D landmarks (InsightFace buffalo_l '1k3d68', frozen) -> Procrustes-aligned shape (204-d) + 15 craniofacial ratios.
#          'clipnm': CLIP ViT-L/14 on the nose-mouth-chin crop of the detected face. Resumable per split. The shape template uses TRAINING faces only.
from insightface.app import FaceAnalysis
APP = FaceAnalysis(name='buffalo_l', root=os.path.join(CACHE, 'insightface'), allowed_modules=['detection', 'landmark_3d_68'], providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
APP.prepare(ctx_id=0, det_size=(320, 320))
assert 'landmark_3d_68' in APP.models, 'buffalo_l landmark_3d_68 model missing - tell Claude'
def load_bgr(p):
    img = cv2.imread(p)
    if img is None:
        img = cv2.cvtColor(np.array(Image.open(p).convert('RGB')), cv2.COLOR_RGB2BGR)
    return img
def face_one(p):
    """-> (68x3 landmarks in image coordinates or None, PIL crop of the nose-mouth-chin region, found)"""
    img = load_bgr(p); h, w = img.shape[:2]; pad = int(0.25 * max(h, w))
    fs = APP.get(cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0))   # padding helps the detector on tight face crops (NB35)
    if fs:
        f = max(fs, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
        L = np.asarray(f.landmark_3d_68, np.float32).copy(); L[:, :2] -= pad
        x0, y0, x1, y1 = [float(v) - pad for v in f.bbox]; found = True
    else:
        L = None; x0, y0, x1, y1 = 0.0, 0.0, float(w), float(h); found = False
    bw, bh = x1 - x0, y1 - y0
    cx0, cx1 = int(max(0, x0 + 0.15 * bw)), int(min(w, x0 + 0.85 * bw)); cy0, cy1 = int(max(0, y0 + 0.40 * bh)), int(min(h, y0 + 1.0 * bh))
    if cx1 - cx0 < 8 or cy1 - cy0 < 8:
        cx0, cy0, cx1, cy1 = 0, 0, w, h
    return L, Image.fromarray(cv2.cvtColor(img[cy0:cy1, cx0:cx1], cv2.COLOR_BGR2RGB)), found
LM, FOUND, NM = {}, {}, {}
for s in SPLITS:
    part = os.path.join(PARTS, 'face_' + s + '.npz')
    if os.path.exists(part):
        d = np.load(part); LM[s], FOUND[s], NM[s] = d['lm'], d['found'], d['clipnm']; continue
    t0 = time.time(); n = len(FACES[s]); lm = np.zeros((n, 68, 3), np.float32); found = np.zeros(n, bool); nm = np.zeros((n, 768), np.float32); buf, bidx = [], []
    for i, p in enumerate(tqdm(FACES[s], desc='faces ' + s)):
        L, crop, ok = face_one(p); found[i] = ok
        if ok:
            lm[i] = L
        buf.append(crop); bidx.append(i)
        if len(buf) == 64 or i == n - 1:
            nm[bidx] = clip_embed(buf); buf, bidx = [], []
    np.savez(part, lm=lm, found=found, clipnm=nm); LM[s], FOUND[s], NM[s] = lm, found, nm
    print('  %s: %d faces in %.0f min' % (s, n, (time.time() - t0) / 60))
# ---- generalised Procrustes: template from the training faces; every shape centred, unit-scaled and rotated onto it ----
def shape_norm(L):
    L = L - L.mean(0); return L / (np.sqrt((L ** 2).sum()) + 1e-9)
def align_to(L, T):
    U, _, Vt = np.linalg.svd(L.T @ T)
    if np.linalg.det(U @ Vt) < 0:
        Vt[-1] *= -1
    return L @ (U @ Vt)
tr_shapes = np.stack([shape_norm(L) for L, ok in zip(LM['train'], FOUND['train']) if ok])
T = shape_norm(tr_shapes.mean(0))
for _ in range(3):
    T = shape_norm(np.stack([align_to(S_, T) for S_ in tr_shapes]).mean(0))
RATIO_NAMES = ['face_w', 'face_h', 'nose_len', 'nose_w', 'mouth_w', 'lip_h', 'upper_lip', 'lower_lip', 'philtrum', 'chin', 'jaw_w', 'eye_w', 'brow_h', 'nose_depth', 'cheek_depth']
def ratios(L):
    """15 scale-free craniofacial ratios (iBUG 68-point indexing), all relative to the inter-ocular distance"""
    d = lambda a, b: float(np.linalg.norm(L[a] - L[b]))
    re_, le_ = L[36:42].mean(0), L[42:48].mean(0); iod = float(np.linalg.norm(re_ - le_)) + 1e-6; brow = L[17:27].mean(0); eyes = 0.5 * (re_ + le_)
    r = [d(0, 16), float(np.linalg.norm(L[8] - brow)), d(27, 33), d(31, 35), d(48, 54), d(51, 57), d(51, 62), d(66, 57), d(33, 51), d(57, 8), d(4, 12),
         0.5 * (d(36, 39) + d(42, 45)), float(np.linalg.norm(brow - eyes)), float(L[30, 2] - 0.5 * (L[31, 2] + L[35, 2])), float(0.5 * (L[1, 2] + L[15, 2]) - L[30, 2])]
    return np.array(r, np.float32) / iod
T_vec = np.r_[T.reshape(-1), ratios(T)]
GEO = {}
for s in SPLITS:
    G = np.zeros((len(LM[s]), 204 + 15), np.float32)
    for i, (L, ok) in enumerate(zip(LM[s], FOUND[s])):
        if ok:
            A = align_to(shape_norm(L), T); G[i] = np.r_[A.reshape(-1), ratios(A)]
        else:
            G[i] = T_vec
    GEO[s] = G
np.savez(os.path.join(FEATDIR, 'geo.npz'), train=GEO['train'], mav=GEO['mav'], extra=GEO['extra'], template=T, found_train=FOUND['train'], found_mav=FOUND['mav'], found_extra=FOUND['extra'])
np.savez(os.path.join(FEATDIR, 'clipnm.npz'), train=NM['train'], mav=NM['mav'], extra=NM['extra'])
for s in SPLITS:
    enote('faces %-6s %6d images | face detected %.1f%%' % (s, len(FOUND[s]), 100 * FOUND[s].mean()))
# ---- sanity: the ratios should differ by sex in the known direction (men: wider face, nose and jaw); identity should be visible in both blocks ----
ytr_c = pd.read_csv(find_one('train_English_voices.csv'), header=None).iloc[:, 192].values.astype(int)
cls2spk = {int(l[5]): l[4] for l in lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
row_g = np.array([spk2g[cls2spk[c]] for c in ytr_c]); R = GEO['train'][:, 204:]; okm = FOUND['train']; gs = sorted(set(row_g))
for j in [0, 2, 3, 4, 10]:
    enote('ratio %-9s ' % RATIO_NAMES[j] + ' | '.join('%s %.3f' % (g, R[okm & (row_g == g), j].mean()) for g in gs))
def gap(E, lab, seed=7):
    rng = np.random.default_rng(seed); idx = rng.choice(len(E), 300, replace=False); Z = E[idx] / (np.linalg.norm(E[idx], axis=1, keepdims=True) + 1e-9); C = Z @ Z.T; l = lab[idx]
    same = (l[:, None] == l[None, :]) & ~np.eye(300, dtype=bool); diff = l[:, None] != l[None, :]
    return C[same].mean(), C[diff].mean()
Gz = (GEO['train'] - GEO['train'].mean(0)) / (GEO['train'].std(0) + 1e-6)
for name, E in [('geo (z-scored)', Gz), ('clipnm', NM['train'])]:
    a, b = gap(E, ytr_c); enote('%-15s sanity: same-id cosine %.3f | different-id %.3f | gap %.3f' % (name, a, b, a - b))
del APP, CLIPM; torch.cuda.empty_cache()


In [ ]:
# Cell 4 [NB39 v1 5-Nov] - VOICE cues. 'whisper_mid': Whisper-large-v3 encoder layers 8-16 (same pooling as our 20-28 block).
#          'wlm_lo' / 'wlm_mid': WavLM-large layers 3-8 / 12-18, mean over time, 12-s centre crop. Frozen public models, declared. Resumable per split.
def run_split(block, s, fn):
    part = os.path.join(PARTS, '%s_%s.npz' % (block, s))
    if os.path.exists(part):
        return dict(np.load(part))
    t0 = time.time(); out = fn(WAVS[s]); np.savez(part, **out); print('  %s %s: %d clips in %.0f min' % (block, s, len(WAVS[s]), (time.time() - t0) / 60)); return out
# ---- Whisper mid layers ----
if 'WENC' not in globals():
    WFE, WENC = get_whisper()
def enc_whisper_mid(paths, bs=8):
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='whisper mid'):
        out.append(whisper_batch(WFE, WENC, paths[i:i + bs], [(8, 17)])[0])
    return {'e': np.concatenate(out).astype(np.float32)}
WHM = {s: run_split('whm', s, enc_whisper_mid)['e'] for s in SPLITS}
del WENC; torch.cuda.empty_cache()
# ---- WavLM-large hidden layers ----
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
try:
    WLFE = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-large')
except Exception:
    WLFE = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
WLM = WavLMModel.from_pretrained('microsoft/wavlm-large').to(dev).eval()
USE_HALF = [True]
def wl_forward(x):
    if USE_HALF[0]:
        try:
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
                return WLM(x, output_hidden_states=True).hidden_states
        except RuntimeError as e:
            print('wavlm fp16 failed (%s) -> fp32 for the rest' % repr(e)[:80]); USE_HALF[0] = False; torch.cuda.empty_cache()
    with torch.no_grad():
        return WLM(x, output_hidden_states=True).hidden_states
def centre(w, sec=12):
    L = 16000 * sec
    if len(w) < 8000:
        w = np.pad(w, (0, 8000 - len(w)))
    return w[(len(w) - L) // 2:(len(w) - L) // 2 + L] if len(w) > L else w
def enc_wavlm(paths):
    lo, mid, bad = [], [], 0
    for p in tqdm(paths, desc='wavlm-large'):
        w = centre(librosa.load(p, sr=16000, mono=True)[0])
        x = WLFE(w, sampling_rate=16000, return_tensors='pt').input_values.to(dev)
        hs = wl_forward(x)
        a = torch.stack(hs[3:9]).float().mean(0)[0].mean(0).cpu().numpy(); b = torch.stack(hs[12:19]).float().mean(0)[0].mean(0).cpu().numpy()
        if not (np.isfinite(a).all() and np.isfinite(b).all()):
            a, b = np.zeros_like(a), np.zeros_like(b); bad += 1
        lo.append(a); mid.append(b)
    if bad:
        enote('wavlm: %d clips gave non-finite outputs (zeroed)' % bad)
    return {'lo': np.stack(lo).astype(np.float32), 'mid': np.stack(mid).astype(np.float32)}
WLO, WMID = {}, {}
for s in SPLITS:
    d = run_split('wlm', s, enc_wavlm); WLO[s], WMID[s] = d['lo'], d['mid']
del WLM; torch.cuda.empty_cache()
for name, D in [('whisper_mid', WHM), ('wlm_lo', WLO), ('wlm_mid', WMID)]:
    np.savez(os.path.join(FEATDIR, name + '.npz'), **D)
    a, b = gap(D['train'], ytr_c); enote('%-11s dim %4d | sanity: same-id cosine %.3f | different-id %.3f | gap %.3f' % (name, D['train'].shape[1], a, b, a - b))


In [ ]:
# Cell 5 - features, labels, CV folds, harness (NB36 Cell 2 verbatim)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# Cell 6 [NB39 v1 5-Nov] - CV: each new block next to system E's inputs (face CLIP | voice R4 + whisper, NAP-8) with E's loss (CE + 2*OPL + 0.5*SupCon L_out, tau 0.07).
# R0_ref = system E's recipe copied from NB36 M0_ref. Stage A usual folds seeds 1-3; stage B NEW folds + seeds 4-6 on the top 3; stage C = all adopted blocks together.
# Training-side only; per-trial scoring; no test statistics. A winner still needs the same block on the two VoxCeleb parts before NB20 can use it.
SUMMARY = list(EXTRACT_NOTES) + SUMMARY
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
BLOCKS = ['geo', 'clipnm', 'whisper_mid', 'wlm_lo', 'wlm_mid']
for name in BLOCKS:
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in ['train', 'mav', 'extra']}
        assert all(len(FEAT[name][s]) == len(FEAT['clip'][s]) for s in ['train', 'mav', 'extra']), name + ' rows do not match - tell Claude'
G_CODE = {g: i for i, g in enumerate(sorted(set(id_gender.values())))}; y_g = np.array([G_CODE[id_gender[c]] for c in y])
fold_of = {c: k for k, fl in enumerate(folds) for c in fl}; grp = np.array([fold_of[c] for c in y])
def gender_probe(X):
    """5-fold identity-disjoint gender accuracy on the challenge rows: how much sex information a block carries"""
    acc = []
    for k in range(K):
        m = grp == k; clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1)).fit(X[~m], y_g[~m]); acc.append((clf.predict(X[m]) == y_g[m]).mean())
    return 100 * np.mean(acc)
for name in BLOCKS:
    if name in FEAT:
        note('block %-11s %4d-d | gender probe %.1f%% (identity-disjoint folds, challenge rows)' % (name, FEAT[name]['train'].shape[1], gender_probe(FEAT[name]['train'])))
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def train_e(F_raw, V_raw, all_tr, vsets, seed, w_al=0.5, tau=0.07):
    """system E's recipe (= NB35 train_lout): 16 ids x 8 clips, CE + 2*OPL + 0.5*L_out, early stopping on the fold's pairs"""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
COMBOS = {'F_geo': (['clip', 'geo'], VOICE), 'F_nm': (['clip', 'clipnm'], VOICE), 'V_whm': (['clip'], VOICE + ['whisper_mid']), 'V_wlo': (['clip'], VOICE + ['wlm_lo']), 'V_wmid': (['clip'], VOICE + ['wlm_mid'])}
missing = [c for c, (fn_, vn_) in COMBOS.items() if any(n not in FEAT for n in fn_ + vn_)]
if missing:
    note('blocks missing, candidates skipped: %s' % missing)
COMBOS = {c: v for c, v in COMBOS.items() if c not in missing}
def inputs(fn_, vn_):
    F_raw = build_train(fn_); V_full = build_train(vn_); U, _ = nap_basis(V_full[N_CH:], NAP_K); return F_raw, nap_apply(V_full, U)
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV39 = os.path.join(RESDIR, 'nb39_cv.csv')
if not os.path.exists(CSV39):
    r = pd.read_csv(os.path.join(RESDIR, 'nb36_cv.csv')); r = r[r.combo == 'M0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    r['combo'] = 'R0_ref'; r.to_csv(CSV39, index=False)
    print('R0_ref copied from NB36 M0_ref (%d runs: stage A %d, stage B %d)' % (len(r), (r.stage == 'A').sum(), (r.stage == 'B').sum()))
done39 = pd.read_csv(CSV39); t0 = time.time()
def run39(stage, names, seeds, fold_data):
    global done39
    for name in names:
        todo = [(seed, k) for seed in seeds for k in range(K) if not ((done39.stage == stage) & (done39.combo == name) & (done39.seed == seed) & (done39.fold == k)).any()]
        if not todo:
            continue
        F_raw, V_raw = inputs(*COMBOS[name])
        for seed in seeds:
            for k in range(K):
                if (seed, k) not in todo:
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_e(F_raw, V_raw, np.concatenate([tr_idx, EXT]), vsets, seed)
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV39, mode='a', header=False, index=False); done39 = pd.concat([done39, row])
            print('%s %-8s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
        del F_raw, V_raw
def tab39(stage, names=None):
    R = pd.read_csv(CSV39); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    if names is not None:
        R = R[R.combo.isin(list(names) + ['R0_ref'])]
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def adopt(TB, c):
    refB = TB.loc['R0_ref']; dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    return (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)), dB, ds, dg
run39('A', list(COMBOS), [1, 2, 3], FOLD_DATA)
TA = tab39('A'); refA = TA.loc['R0_ref']; TOP = [c for c in TA.index if c != 'R0_ref'][:3]
run39('B', TOP, [4, 5, 6], FOLD_B)
TB = tab39('B')
note('NB39 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe from NB36):'); note(TA.to_string())
note('NB39 STAGE B (NEW folds + NEW seeds 4-6, top 3 of stage A):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
WIN = []
for c in TOP:
    ok, dB, ds, dg = adopt(TB, c)
    note('%-8s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, '+'.join(n for n in COMBOS[c][0] + COMBOS[c][1] if n not in ['clip'] + VOICE), TA.loc[c, 'overall'] - refA['overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    if ok:
        WIN.append(c)
if len(WIN) >= 2:
    fn_ = ['clip'] + sorted({n for c in WIN for n in COMBOS[c][0] if n != 'clip'}); vn_ = VOICE + sorted({n for c in WIN for n in COMBOS[c][1] if n not in VOICE})
    name = 'C_' + '+'.join(WIN); COMBOS[name] = (fn_, vn_)
    run39('A', [name], [1, 2, 3], FOLD_DATA); run39('B', [name], [4, 5, 6], FOLD_B)
    TAc, TBc = tab39('A', [name]), tab39('B', [name]); ok, dB, ds, dg = adopt(TBc, name)
    note('STAGE C %s (face %s | voice %s): A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (name, '+'.join(fn_), '+'.join(vn_), TAc.loc[name, 'overall'] - TAc.loc['R0_ref', 'overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    for c in WIN:
        note('  vs %s alone on stage B: %+.2f' % (c, TBc.loc[name, 'overall'] - TB.loc[c, 'overall']))
elif len(WIN) == 1:
    note('one adopted block (%s); no combination to test' % WIN[0])
else:
    note('nothing adopted; no combination to test')
note('NEXT if anything is adopted: the same block(s) must be extracted on VoxCeleb parts vox2_test_mp4 and vox2_mp4_1 (new notebook) before NB20 v9 can train system H.')
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
